<a href="https://colab.research.google.com/github/andrelison/salesinsight-py/blob/feat%2Fpipeline-dados/salesinsigtht.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa 1: dados e inspeção (RF01 e RF02)

In [1]:
import json
import os
import random
import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter


In [2]:
# RF01 - CRIAR OU CARREGAR O DATASET


def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


In [3]:
# RF02 - INSPECIONAR E DESCREVER OS DADOS

def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

In [4]:
# =============================================================================
# RF03 - TRATAR OS DADOS (DATETIME)
# =============================================================================

def tratar_dados(df):
    """
    Trata o DataFrame de vendas (RF03).
    Retorna: (df_limpo, relatorio), onde relatorio e um dicionario
    com as contagens de registros iniciais, convertidos e finais.
    """
    df = df.copy()
    antes = df.copy()                      # copia para comparar depois
    registros_iniciais = len(df)

    # 1. remover espacos extras nas colunas de texto
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto:
        df[col] = df[col].str.strip()
    # True nas linhas em que algum texto mudou depois do strip
    mudou_texto = (
        antes[colunas_texto].fillna("") != df[colunas_texto].fillna("")
    ).any(axis=1)

        # verificacao informativa com regex: nomes de cliente fora do padrao
    # (apenas conta e informa, nenhum registro e alterado ou removido)
    clientes_fora_do_padrao = int(
        (~df["cliente"].apply(lambda c: bool(re.match(r"^Cliente_\d{3}$", str(c))))).sum()
    )

    # 2. converter data_venda e descartar datas invalidas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    datas_invalidas = int(df["data_venda"].isna().sum())
    df = df[df["data_venda"].notna()].copy()
    datas_convertidas = len(df)

    # 3. ajustar tipos numericos
    tipos_ajustados = 0
    for coluna, tipo in {"quantidade": int, "preco_unitario": float}.items():
        convertida = df[coluna].astype(tipo)
        if convertida.dtype != antes[coluna].dtype:
            tipos_ajustados += 1
        df[coluna] = convertida

# 4. montar e imprimir o relatorio de limpeza
    # "sem alteracao" = linhas cujo texto e tipos numericos nao mudaram
    if tipos_ajustados == 0:
        sem_alteracao = int((~mudou_texto.loc[df.index]).sum())
    else:
        sem_alteracao = 0

    relatorio = {
        "registros_iniciais": registros_iniciais,
        "espacos_removidos (linhas)": int(mudou_texto.sum()),
        "datas_convertidas (texto -> datetime)": datas_convertidas,
        "datas_invalidas_removidas": datas_invalidas,
        "colunas_numericas_ajustadas": tipos_ajustados,
        "clientes_fora_do_padrao (informativo)": clientes_fora_do_padrao,
        "registros_sem_alteracao": sem_alteracao,
        "registros_finais": len(df),
    }


    print("\n=== RELATORIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")

    return df.reset_index(drop=True), relatorio


